# L5 — Knapsack: the first real constraint

**2026 Quantum Accelerator · Day 1 · QUBO ladder · step 5 of 5 · ~20 minutes**

**New on this step:** a *global inequality*. A QUBO cannot say "≤", so the rule becomes an
equality with a **slack variable**, the slack is written in **bits**, and the equality is
**squared** into a penalty. Auxiliary variables appear that are not part of the answer; Q becomes
dense and dominated by the penalty. This is the **encoding tax**, in its smallest possible
instance; notebook 04 is what it does at scale, and §5b below sweeps the penalty on a 40-item bag.

**The problem.** Three items, a bag with a weight limit, most value that fits. **Why it matters.**
Budgets, cargo, capacity — every "at most" rule in business. **Plan.** The slide instance, then a 40-item bag
(the one the archived notebook 02 used).

---
## 1. The slide instance

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import itertools
import helpers_ladder as h
import helpers_knapsack as hk

weights, values, C, P, coeffs = h.knapsack_slide()   # weights (2,3,4), values (6,4,5), C = 7, P = 7
C = 7                                                # <-- CHANGE ME: 6 or 5 (slack bits are no longer powers of two)
coeffs = h.slack_bits(C)
item_labels = ["A", "B", "C"]
print(f"slack coefficients for C = {C}: {coeffs.tolist()}")

print(f"\n{'item':<6}{'weight':>8}{'value':>7}")
for l, wt, v in zip(item_labels, weights, values):
    print(f"{l:<6}{wt:>8}{v:>7}")
print(f"\n{'packing':<10}{'weight':>8}{'value':>7}  fits")
for bits in itertools.product([0, 1], repeat=3):
    chk = h.knapsack_check(values, weights, C, bits)
    name = "".join(l for l, b in zip(item_labels, bits) if b) or "-"
    print(f"{name:<10}{chk['weight']:>8}{chk['value']:>7}  {chk['feasible']}")

slack coefficients for C = 7: [1, 2, 4]

item    weight  value
A            2      6
B            3      4
C            4      5

packing     weight  value  fits
-                0      0  True
C                4      5  True
B                3      4  True
BC               7      9  True
A                2      6  True
AC               6     11  True
AB               5     10  True
ABC              9     15  False


**What you see.** Greedy by value per kg takes A (3.0) then B (1.3): value 10. The optimum is
A and C: weight 6, value 11. At the optimum there is 1 kg of unused room — remember that number.

**Step 1 — make the inequality an equality.** `weight + S = 7`, with a slack `S ∈ {0, …, 7}`
(the room that is left).

**Step 2 — write the slack in bits.** `S = s₁ + 2 s₂ + 4 s₄`: three new binary variables that
can represent exactly 0 to 7.

**Step 3 — square it into a penalty.** Minimise `−value + P·(weight + S − 7)²`, with
`P > max value = 6`. The penalty is 0 exactly when the bag is full up to the slack.

---
## 2. Q by hand

In [2]:
labels = ["x1", "x2", "x3"] + [f"s{c}" for c in coeffs] if len(set(coeffs)) == len(coeffs) \
    else ["x1", "x2", "x3"] + [f"s{k+1}" for k in range(len(coeffs))]
a = np.concatenate([weights, coeffs]).astype(float)       # (2,3,4, 1,2,4)
v = np.concatenate([values, np.zeros(len(coeffs))])
Q = P * np.outer(a, a)
np.fill_diagonal(Q, -v + P * (a * a - 2 * C * a))
CONST = P * C * C                                          # 343
h.print_matrix(Q, labels, f"Q (P = {P}),  constant = {CONST}")
hk.show_qubo_matrix(Q, 3)

Q (P = 7),  constant = 343
          x1    x2    x3    s1    s2    s4
    x1  -174    42    56    14    28    56
    x2    42  -235    84    21    42    84
    x3    56    84  -285    28    56   112
    s1    14    21    28   -91    14    28
    s2    28    42    56    14  -168    56
    s4    56    84   112    28    56  -280


**What you see.** Every pair interacts, including slack–slack: the heat map shows the item block
(top left) and the slack block (bottom right) both filled. The largest entry is 285 against item
values of 4 to 6 — the penalty dominates the matrix.

---
## 3. Every answer

In [3]:
table = h.exact_table(Q, labels, const=CONST, top=8)
table["weight"] = [int(weights @ r.values[:3]) for _, r in table.iterrows()]
table["value"] = [int(values @ r.values[:3]) for _, r in table.iterrows()]
table["S"] = [int(coeffs @ r.values[3:3 + len(coeffs)]) for _, r in table.iterrows()]
table

,x1,x2,x3,s1,s2,s4,energy,objective,weight,value,S
0,1,0,1,1,0,0,-354,-11,6,11,1
1,1,1,0,0,1,0,-353,-10,5,10,2
2,0,1,1,0,0,0,-352,-9,7,9,0
3,1,0,0,1,0,1,-349,-6,2,6,5
4,0,0,1,1,1,0,-348,-5,4,5,3
5,0,1,0,0,0,1,-347,-4,3,4,4
6,1,0,1,0,0,0,-347,-4,6,11,0
7,1,0,1,0,1,0,-347,-4,6,11,2


**What you see.** The best row is A and C with objective −11, weight 6 and slack `S = 1`. The
slack bits are *chosen by the solver* to make the penalty exactly zero, and then thrown away:
only `x1..x3` are the answer. The first infeasible row (all three items) costs
`−15 + 7·(2 + S)² ≥ 13`, far above the optimum.

---
## 4. The same Q from PyQUBO

In [4]:
from pyqubo import Array, Constraint, Placeholder

M = len(coeffs)
x = Array.create("x", shape=3, vartype="BINARY"); s = Array.create("s", shape=M, vartype="BINARY")
objective = -sum(int(vi) * xi for vi, xi in zip(values, x))
load  = sum(int(wi) * xi for wi, xi in zip(weights, x))
slack = sum(int(c) * si for c, si in zip(coeffs, s))
model = (objective + Placeholder("lam") * Constraint((load + slack - C) ** 2, label="capacity")).compile()
qubo, offset = model.to_qubo(feed_dict={"lam": P})
labels_lib = [f"x[{k}]" for k in range(3)] + [f"s[{k}]" for k in range(M)]
print(np.allclose(Q, h.symmetric_from_qubo_dict(qubo, labels_lib)), offset)   # True 343.0

True 343.0


In [5]:
import dimod
for lam in (1, 7):                                   # <-- CHANGE ME: try 2
    bqm = model.to_bqm(feed_dict={"lam": lam})
    ss = dimod.ExactSolver().sample(bqm)
    decoded = model.decode_sampleset(ss, feed_dict={"lam": lam})
    best = sorted(decoded, key=lambda d: d.energy)[:2]
    for d in best:
        items = [n for i, n in enumerate("ABC") if d.array("x", i) == 1]
        print(f"lam={lam}: {items!s:<14} energy {d.energy:7.1f}   broken: {d.constraints(only_broken=True)}")

lam=1: ['A', 'B', 'C'] energy   -11.0   broken: {'capacity': (False, 4.0)}
lam=1: ['A', 'C']     energy   -11.0   broken: {}
lam=7: ['A', 'C']     energy   -11.0   broken: {}
lam=7: ['A', 'B']     energy   -10.0   broken: {}


**What you see.** The solver does not hand back "a mysteriously bad number". The named
`Constraint("capacity")` reports *which rule* broke and by how much: the value 4.0 is the squared
excess, (9 − 7)² = 4. PyQUBO's energies already include the constant 343, so they read as the
slide's objective. At `lam = 1` the illegal packing A + B + C ties the legal A + C at −11; at
`lam = 7` it is gone, and the runner-up is the legal A + B at −10.

**The penalty dial on the slide instance.**

In [6]:
for P_try in (1, 2, 3, 7, 20):
    Qp, cp, _ = h.knapsack_Q(values, weights, C, P_try, coeffs)
    t = h.exact_table(Qp, labels, const=cp)
    best = t[t.energy == t.energy.iloc[0]]
    for _, r in best.iterrows():
        chk = h.knapsack_check(values, weights, C, r.values[:3])
        print(f"P = {P_try:>2}: argmin {tuple(int(b) for b in r.values[:3 + M])}  objective {r['objective']}  "
              f"weight {chk['weight']}  fits {chk['feasible']}")

P =  1: argmin (1, 0, 1, 1, 0, 0)  objective -11  weight 6  fits True
P =  1: argmin (1, 1, 1, 0, 0, 0)  objective -11  weight 9  fits False
P =  2: argmin (1, 0, 1, 1, 0, 0)  objective -11  weight 6  fits True
P =  3: argmin (1, 0, 1, 1, 0, 0)  objective -11  weight 6  fits True
P =  7: argmin (1, 0, 1, 1, 0, 0)  objective -11  weight 6  fits True
P = 20: argmin (1, 0, 1, 1, 0, 0)  objective -11  weight 6  fits True


**What you see.** At P = 1 there is a tie at −11 between the legal A + C and the *illegal*
A + B + C (value 15, excess 2, penalty 4). From P = 2 the answer is unique and legal. On this
instance P = 2 already works and P = 7 is the provable bound; the gap between "works here" and
"provably works" is what a λ sweep explores.

---
## 5. Now too big for a slide: the forty-item bag of notebook 02

In [7]:
import time

values_b, weights_b, C_b = h.knapsack_m2_instance()            # rng 7, 40 items, C = 525
t0 = time.time(); opt = h.knapsack_dp(values_b, weights_b, C_b); t_dp = time.time() - t0
coeffs_b = h.slack_bits(C_b)                                   # 10 slack bits, the last one is 14
Q_b, CONST_b, _ = h.knapsack_Q(values_b, weights_b, C_b, P=2, coeffs=coeffs_b)   # <-- CHANGE ME: P
ss, dt = h.anneal(Q_b)
stats = h.knapsack_stats(ss, values_b, weights_b, C_b, 40)

print(f"{len(values_b)} items + {len(coeffs_b)} slack bits = {len(values_b) + len(coeffs_b)} variables")
print(f"DP optimum {opt}   ({t_dp:.3f} s)")
print(f"SA: {stats['feasible_pct']:.0f} % feasible, best feasible value {stats['best_value']}, "
      f"gap {stats['gap_pct']:.2f} %   ({dt:.2f} s)")
hk.show_qubo_matrix(Q_b, 40)

40 items + 10 slack bits = 50 variables
DP optimum 1789   (0.000 s)
SA: 86 % feasible, best feasible value 1700, gap 4.97 %   (0.19 s)


**What you see.** 40 items became 50 variables, and a dense 50 × 50 matrix whose entries reach
the hundreds of thousands against item values below 100. At P = 2 about 86 % of the reads respect
the capacity and the best legal bag is about 5 % short of the DP optimum.

---
## 5b. The penalty dial

In [8]:
rows = h.penalty_sweep_knapsack(values_b, weights_b, C_b, Ps=(0.1, 0.3, 1, 2, 5, 20, 100))   # <-- CHANGE ME
print(f"{'P':>6}{'feasible':>10}{'best':>7}{'gap':>8}")
for r in rows:
    print(f"{r['P']:>6}{r['feasible_pct']:>9.0f}%{str(r['best_value']) if r['best_value'] else '-':>7}"
          f"{('%.1f%%' % r['gap_pct']) if r['gap_pct'] is not None else '-':>8}")
h.plot_penalty_sweep(rows, reference=opt, xlabel="penalty P (= lambda in notebook 02)")

     P  feasible   best     gap
   0.1       12%   1785    0.2%
   0.3       52%   1775    0.8%
     1       76%   1653    7.6%
     2       86%   1700    5.0%
     5       96%   1469   17.9%
    20       98%   1495   16.4%
   100      100%   1425   20.3%


**What you see.** Feasibility rises with `P` while quality falls, and they never peak together.
The details differ from the λ table in `README.md` (notebook 02 builds the model through PyQUBO
with string labels, which changes the annealer's variable order and random stream), but the shape
is identical. The full 40-item sweep is this section; the archived notebook 02 is the same experiment through PyQUBO's `feed_dict`.

---
## 6. What changed on this step

- A global "≤" became an equality with a slack, the slack became bits, the equality became a square.
- Cost of that trick at scale: 10 extra variables, a dense 50 × 50 matrix, coefficients in the
  hundreds of thousands, and a parameter that trades legality against quality.
- The slack bits are not part of the answer: only the item variables are read out.
- Hand-offs: **04** (what happens with 200
  constraints), **03** (independent sets at railway scale), **05** (where there is no constraint and QUBO wins).
- Solvers that take the inequality natively (CP-SAT today; SQBM+ `qplib`, D-Wave CQM) are the exit from the encoding tax.

### Rules of thumb you can take home

- A λ larger than the largest profit is provably sufficient for knapsack-type rules (Lucas 2014, §5.2).
- Computable penalty bounds exist, so this is not guesswork (Glover, Kochenberger & Du 2019).
- Unbalanced penalisation avoids slack bits for inequalities, at the price of two coefficients
  (Montañez-Barrera et al. 2024).
- *Low λ is a lottery ticket, high λ is an insurance policy — they never peak together.*

---
## Stretch

**A.** Count slack bits for C = 525, 5 250, 52 500: the encoding grows like log C, the
coefficients like C². Which one hurts an annealer more?

**B.** Unary slack (C + 1 bits, at most one set): more variables, smaller coefficients. Build it for
the slide instance and compare the largest |Q| entry.

**C.** Add the rule *exactly two items* as a second squared penalty `μ(x₁ + x₂ + x₃ − 2)²`. With
`μ = 1` the solver takes three items anyway (cheap fine, broken on purpose); with `μ = 100` it
obeys. Which family of the backbone slide is this, and why is it *free* in variables while the
capacity rule cost three?

**D.** Drop the slack entirely and penalise only the excess with `max(0, weight − C)`. Why is that
not a QUBO, and what do solvers with native inequalities do instead?